# Otimização Optuna

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime
import pandas as pd
from src.Classification.Optimizer import ClassificationOptunaOptimizer
from src.Data.Processor import DataStreamProcessor

# Definição dos datasets
# categorias = ['Consistência', 'Generalização', 'Adaptação', 'Recorrência']
# tamanhos = ['25', '200', '1000']

categorias = ['Generalização']
tamanhos = ['1000']

datasets = [f'data/15k/{cat}/{cat}_{tam}.csv' for cat in categorias for tam in tamanhos]

FEATURES = [
    'Fwd Packet Length Min',
    'Total Fwd Packets',
    'Fwd Packet Length Max',
    'Packet Length Variance',
    'Init_Win_bytes_forward',
    'Flow IAT Mean',
    'Fwd Packets/s',
    'Fwd Packet Length Std',
    'Flow Duration',
    'Total Backward Packets',
    'URG Flag Count',
    'Init_Win_bytes_backward',
    'Flow IAT Min',
    'Bwd Packets/s',
    'Bwd IAT Mean',
    'Down/Up Ratio',
    'Bwd IAT Min',
    'Bwd Packet Length Mean',
    'Bwd Packet Length Max',
    'Fwd Header Length',
    'Total Length of Fwd Packets',
    'ACK Flag Count',
    'Active Mean',
    'Fwd Packet Length Mean',
    'Fwd PSH Flags',
]

In [ ]:
id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando otimização para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    processor = DataStreamProcessor(logging=False, selected_features=FEATURES)
    
    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    optimizer = ClassificationOptunaOptimizer(
        stream=stream,
        n_trials=3,
        target_names=targets,
        n_runs=3
    )
    
    best_params = optimizer.optimize(
        model_name='HAT',
        warmup_instances=0,
        experiment_name=nome_experimento,
        num_features=len(features),
        exec_id=id_execucao,
        window_evaluation=100
    )

# Execução Default

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime
import pandas as pd
from src.Classification.Pipeline import ClassificationExperimentRunner
from src.Classification.Models import get_classification_models
from src.Data.Processor import DataStreamProcessor

categorias = ['Consistência', 'Generalização', 'Adaptação', 'Recorrência']
tamanhos = ['25', '200', '1000']

# categorias = ['Generalização']
# tamanhos = ['1000']

datasets = [f'data/15k/{cat}/{cat}_{tam}.csv' for cat in categorias for tam in tamanhos]

FEATURES = [
    'Fwd Packet Length Min',
    'Total Fwd Packets',
    'Fwd Packet Length Max',
    'Packet Length Variance',
    'Init_Win_bytes_forward',
    'Flow IAT Mean',
    'Fwd Packets/s',
    'Fwd Packet Length Std',
    'Flow Duration',
    'Total Backward Packets',
    'URG Flag Count',
    'Init_Win_bytes_backward',
    'Flow IAT Min',
    'Bwd Packets/s',
    'Bwd IAT Mean',
    'Down/Up Ratio',
    'Bwd IAT Min',
    'Bwd Packet Length Mean',
    'Bwd Packet Length Max',
    'Fwd Header Length',
    'Total Length of Fwd Packets',
    'ACK Flag Count',
    'Active Mean',
    'Fwd Packet Length Mean',
    'Fwd PSH Flags',
]

## Hoeffding Tree (HT)

### Todas as características

In [ ]:
id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

parametros_ht = {
    'grace_period': 200,
    'confidence': 1e-3,
    'tie_threshold': 0.05
}

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando avaliação para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    processor = DataStreamProcessor(logging=False)

    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    algoritmos = get_classification_models(
        stream.get_schema(),
        selected_models=['HT'],
        ht_params=parametros_ht
    )
    
    runner = ClassificationExperimentRunner(target_names=targets, n_runs=5)
    
    runner.run_classification_evaluation(
        stream=stream,
        algorithms=algoritmos,
        window_evaluation=100,               
        warmup_instances=0,
        title=nome_experimento,              
        algorithm_params=parametros_ht,     
        is_optimized=False,
        num_features=len(features),
        exec_id=id_execucao              
    )

### Melhores características

In [ ]:
id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

parametros_ht = {
    'grace_period': 200,
    'confidence': 1e-3,
    'tie_threshold': 0.05
}

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando avaliação para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    processor = DataStreamProcessor(logging=False, selected_features=FEATURES)

    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    algoritmos = get_classification_models(
        stream.get_schema(),
        selected_models=['HT'],
        ht_params=parametros_ht
    )
    
    runner = ClassificationExperimentRunner(target_names=targets, n_runs=5)
    
    runner.run_classification_evaluation(
        stream=stream,
        algorithms=algoritmos,
        window_evaluation=100,               
        warmup_instances=0,
        title=nome_experimento,              
        algorithm_params=parametros_ht,     
        is_optimized=False,
        num_features=len(features),
        exec_id=id_execucao              
    )

## Adaptive Random Forest Classifier (ARF)

### Todas as características

In [ ]:
id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

parametros_arf = {
    'base_learner': None,
    'ensemble_size': 100, 
    'max_features': 0.6, 
    'lambda_param': 6.0,
    'minibatch_size': None, 
    'number_of_jobs': 1, 
    'drift_detection_method': None,
    'warning_detection_method': None, 
    'disable_weighted_vote': False,
    'disable_drift_detection': False, 
    'disable_background_learner': False
}

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando avaliação para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    processor = DataStreamProcessor(logging=False)

    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    algoritmos = get_classification_models(
        stream.get_schema(),
        selected_models=['ARF'],
        ht_params=parametros_arf
    )
    
    runner = ClassificationExperimentRunner(target_names=targets, n_runs=5)
    
    runner.run_classification_evaluation(
        stream=stream,
        algorithms=algoritmos,
        window_evaluation=100,               
        warmup_instances=0,
        title=nome_experimento,              
        algorithm_params=parametros_arf,     
        is_optimized=False,
        num_features=len(features),
        exec_id=id_execucao              
    )

### Melhores características

In [ ]:
id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

parametros_arf = {
    'base_learner': None,
    'ensemble_size': 100, 
    'max_features': 0.6, 
    'lambda_param': 6.0,
    'minibatch_size': None, 
    'number_of_jobs': 1, 
    'drift_detection_method': None,
    'warning_detection_method': None, 
    'disable_weighted_vote': False,
    'disable_drift_detection': False, 
    'disable_background_learner': False
}

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando avaliação para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    processor = DataStreamProcessor(logging=False, selected_features=FEATURES)

    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    algoritmos = get_classification_models(
        stream.get_schema(),
        selected_models=['ARF'],
        ht_params=parametros_arf
    )
    
    runner = ClassificationExperimentRunner(target_names=targets, n_runs=5)
    
    runner.run_classification_evaluation(
        stream=stream,
        algorithms=algoritmos,
        window_evaluation=100,               
        warmup_instances=0,
        title=nome_experimento,              
        algorithm_params=parametros_arf,     
        is_optimized=False,
        num_features=len(features),
        exec_id=id_execucao              
    )

## Leveraging Bagging (LB)

### Todas as características

In [ ]:
id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

parametros_lb = {
    'base_learner': None,
    'ensemble_size': 100, 
    'minibatch_size': None, 
    'number_of_jobs': None
}

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando avaliação para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    processor = DataStreamProcessor(logging=False)

    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    algoritmos = get_classification_models(
        stream.get_schema(),
        selected_models=['LB'],
        ht_params=parametros_lb
    )
    
    runner = ClassificationExperimentRunner(target_names=targets, n_runs=5)
    
    runner.run_classification_evaluation(
        stream=stream,
        algorithms=algoritmos,
        window_evaluation=100,               
        warmup_instances=0,
        title=nome_experimento,              
        algorithm_params=parametros_lb,     
        is_optimized=False,
        num_features=len(features),
        exec_id=id_execucao              
    )

### Melhores características

In [ ]:
id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

parametros_lb = {
    'base_learner': None,
    'ensemble_size': 100, 
    'minibatch_size': None, 
    'number_of_jobs': None
}

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando avaliação para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    processor = DataStreamProcessor(logging=False, selected_features=FEATURES)

    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    algoritmos = get_classification_models(
        stream.get_schema(),
        selected_models=['LB'],
        ht_params=parametros_lb
    )
    
    runner = ClassificationExperimentRunner(target_names=targets, n_runs=5)
    
    runner.run_classification_evaluation(
        stream=stream,
        algorithms=algoritmos,
        window_evaluation=100,               
        warmup_instances=0,
        title=nome_experimento,              
        algorithm_params=parametros_lb,     
        is_optimized=False,
        num_features=len(features),
        exec_id=id_execucao              
    )

## Hoeffding Adaptive Tree (HAT)

### Todas as características

In [ ]:
id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

parametros_hat = {
    'grace_period': 200,
    'confidence': 1e-3,
    'tie_threshold': 0.05
}

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando avaliação para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    processor = DataStreamProcessor(logging=False)

    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    algoritmos = get_classification_models(
        stream.get_schema(),
        selected_models=['HAT'],
        ht_params=parametros_hat
    )
    
    runner = ClassificationExperimentRunner(target_names=targets, n_runs=5)
    
    runner.run_classification_evaluation(
        stream=stream,
        algorithms=algoritmos,
        window_evaluation=100,               
        warmup_instances=0,
        title=nome_experimento,              
        algorithm_params=parametros_hat,     
        is_optimized=False,
        num_features=len(features),
        exec_id=id_execucao              
    )

### Melhores características

In [ ]:
id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

parametros_hat = {
    'grace_period': 200,
    'confidence': 1e-3,
    'tie_threshold': 0.05
}

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando avaliação para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    processor = DataStreamProcessor(logging=False, selected_features=FEATURES)

    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    algoritmos = get_classification_models(
        stream.get_schema(),
        selected_models=['HAT'],
        ht_params=parametros_hat
    )
    
    runner = ClassificationExperimentRunner(target_names=targets, n_runs=5)
    
    runner.run_classification_evaluation(
        stream=stream,
        algorithms=algoritmos,
        window_evaluation=100,               
        warmup_instances=0,
        title=nome_experimento,              
        algorithm_params=parametros_hat,     
        is_optimized=False,
        num_features=len(features),
        exec_id=id_execucao              
    )